In [ ]:
import os
import pandas as pd
import numpy as np
import openpyxl
from sklearn.linear_model import LinearRegression

In [ ]:
""" 預測IC值 """
""" 讀取 Excel 檔案 """
workbook = openpyxl.load_workbook("OLS.xlsx")
sheet = workbook["預測IC"]

""" 預測 bm 的 IC """
df_x = pd.read_excel('bm.xlsx', sheet_name="bm補值")
df_y = pd.read_excel('bm.xlsx', sheet_name="bmIC")
df_x = df_x.T
df_y = df_y.T
df_x = df_x.iloc[2:,:]
df_y = df_y.iloc[2:,:]

reg = LinearRegression()
n1, n2 = 166, 300
for n in range(n1, n2):
    X = df_x.iloc[0:n,:].values
    Y = df_y.iloc[1:n+1,:].values
    test_X = df_x.iloc[n:n+1,:].values
    reg.fit(X, Y)
    predIC = reg.predict(test_X)
    sheet.cell(row=3, column=n-n1+3, value=predIC[0,0])

""" 預測 size 的 IC """
df_x = pd.read_excel('size.xlsx', sheet_name="size補值")
df_y = pd.read_excel('size.xlsx', sheet_name="sizeIC")
df_x = df_x.T
df_y = df_y.T
df_x = df_x.iloc[2:,:]
df_y = df_y.iloc[2:,:]
reg = LinearRegression()
n1, n2 = 166, 300
for n in range(n1, n2):
    X = df_x.iloc[0:n,:].values
    Y = df_y.iloc[1:n+1,:].values
    test_X = df_x.iloc[n:n+1,:].values
    reg.fit(X, Y)
    predIC = reg.predict(test_X)
    sheet.cell(row=4, column=n-n1+3, value=predIC[0,0])

""" 預測 mom 的 IC """
df_x = pd.read_excel('mom.xlsx', sheet_name="mom補值")
df_y = pd.read_excel('mom.xlsx', sheet_name="momIC")
df_x = df_x.T
df_y = df_y.T
df_x = df_x.iloc[2:,:]
df_y = df_y.iloc[2:,:]

reg = LinearRegression()
n1, n2 = 154, 288
for n in range(n1, n2):
    X = df_x.iloc[0:n,:].values
    Y = df_y.iloc[1:n+1,:].values
    test_X = df_x.iloc[n:n+1,:].values
    reg.fit(X, Y)
    predIC = reg.predict(test_X)
    sheet.cell(row=5, column=n-n1+3, value=predIC[0,0])

""" 存檔 """
workbook.save("OLS.xlsx")

In [ ]:
"""  讀入數據  """
IC = pd.read_excel('OLS.xlsx', sheet_name="預測IC").T
IC = np.array(IC.iloc[2:,1:4])

# 讀取 bm（bm 補值＋下個月月報酬補值）
bm = pd.read_excel('bm.xlsx', sheet_name="bm補值").T
# 2013/12 ~ 2024/12
bm = np.array(bm.iloc[169:,:])
bm_ret = pd.read_excel('bm.xlsx', sheet_name="下個月月報酬補值").T
bm_ret = np.array(bm_ret.iloc[169:,:])

# 讀取 size（size 補值＋下個月月報酬補值）
size = pd.read_excel('size.xlsx', sheet_name="size補值").T
# 2013/12 ~ 2024/12
size = np.array(size.iloc[169:,:])
size_ret = pd.read_excel('size.xlsx', sheet_name="下個月月報酬補值").T
size_ret = np.array(size_ret.iloc[169:,:])

# 讀取 mom（mom 補值＋下個月月報酬補值）
mom = pd.read_excel('mom.xlsx', sheet_name="mom補值").T
# 2013/12 ~ 2024/12
mom = np.array(mom.iloc[157:,:])
mom_ret = pd.read_excel('mom.xlsx', sheet_name="下個月月報酬補值").T
mom_ret = np.array(mom_ret.iloc[157:,:])

ID = pd.read_excel('bm.xlsx', sheet_name="bm補值")
ID = np.array(ID.iloc[:,0:2])

"""  計算投資組合  """
# 2013/12 ~ 2024/12
for n in range(133):
    c = pd.DataFrame()
    sel_index = np.argmax(abs(IC[n]))
    sel_value = IC[n][sel_index]

    # 選擇因子與報酬
    if sel_index == 0:
        fac, sel, ret = bm[n], 'bm', bm_ret[n]
    elif sel_index == 1:
        fac, sel, ret = size[n], 'size', size_ret[n]
    elif sel_index == 2:
        fac, sel, ret = mom[n], 'mom', mom_ret[n]

    c['factor'] = fac
    c['return'] = ret
    c['rank'] = c['factor'].rank(method='max')

    c = c.sort_values(by='rank')

    row = [12,13,14,15,16,17,18,19,20,21,22,23,24,25]
    group = [185,92,37,18,10,9,8,7,6,5,4,3,2,1]

    for r, g in zip(row, group):
        high = np.mean(c['return'].iloc[-g:])
        low = np.mean(c['return'].iloc[:g])
        ans = high - low if sel_value > 0 else low - high
        sheet.cell(row=r, column=n+3, value=ans)

    d = c.index.tolist()  # 將 DataFrame 的索引轉換成 Python list
    d1 = d[0]     # 排名最低的股票
    d2 = d[1847]  # 或 d[-1]，排名最高的股票（共 1848 檔股票）
    a1 = ID[d1]   # 利用原始 index 到 ID 陣列中查找對應公司 ID/名稱
    a2 = ID[d2]   # 對應 ID 的資料
    # 寫入這兩檔股票的 ID 與名稱到 Excel 中對應列
    sheet.cell(row=28, column=n+3, value=a1[0])
    sheet.cell(row=29, column=n+3, value=a1[1])
    sheet.cell(row=30, column=n+3, value=a2[0])
    sheet.cell(row=31, column=n+3, value=a2[1])

"""  存檔  """
workbook.save('OLS.xlsx')